# Movie Recommender System

**Content-Based Filtering** using TF-IDF vectorization and cosine similarity.

Features used: overview · genres · keywords · top cast · director

In [ ]:
import numpy as np
import pandas as pd
import ast
import pickle
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


import os
os.makedirs('models', exist_ok=True)
print('All imports successful')


## Section 1 - Load Data

In [ ]:
movies  = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

print(f'Movies  : {movies.shape}')
print(f'Credits : {credits.shape}')

movies = movies.merge(credits, on='title')
print(f'Merged  : {movies.shape}')

# Keep only relevant columns
movies = movies[[
    'movie_id', 'title', 'overview',
    'genres', 'keywords', 'cast', 'crew',
    'vote_average', 'vote_count', 'popularity', 'release_date'
]]
movies.head(2)


## Section 2 - Exploratory Data Analysis

In [ ]:
print('Dataset Info:')
print(f'  Total movies     : {len(movies)}')
print(f'  Missing values   :\n{movies.isnull().sum()}')
print(f'  Avg rating       : {movies["vote_average"].mean():.2f}')
print(f'  Rating range     : {movies["vote_average"].min()} - {movies["vote_average"].max()}')

# Drop nulls
movies.dropna(inplace=True)
movies.drop_duplicates(subset='title', inplace=True)
print(f'\nAfter cleaning    : {len(movies)} movies')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Rating distribution
axes[0].hist(movies['vote_average'], bins=30, color='#667eea', edgecolor='white')
axes[0].set_title('Rating Distribution', fontweight='bold')
axes[0].set_xlabel('Vote Average')
axes[0].set_ylabel('Count')

# Popularity distribution (log scale)
axes[1].hist(movies['popularity'], bins=40, color='#764ba2', edgecolor='white')
axes[1].set_title('Popularity Distribution', fontweight='bold')
axes[1].set_xlabel('Popularity')
axes[1].set_yscale('log')

# Top 10 most popular movies
top10 = movies.nlargest(10, 'popularity')[['title', 'popularity']]
axes[2].barh(top10['title'], top10['popularity'], color='#f093fb')
axes[2].set_title('Top 10 Most Popular', fontweight='bold')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('models/eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA plot saved')


## Section 3 - Feature Engineering

In [ ]:
def parse_list(obj):
    """Parse JSON-like list fields and return list of names."""
    try:
        return [item['name'] for item in ast.literal_eval(obj)]
    except:
        return []

def parse_top_cast(obj, top_n=5):
    """Return top N cast names."""
    try:
        return [item['name'] for item in ast.literal_eval(obj)][:top_n]
    except:
        return []

def parse_director(obj):
    """Extract director name from crew list."""
    try:
        for item in ast.literal_eval(obj):
            if item['job'] == 'Director':
                return [item['name']]
        return []
    except:
        return []

movies['genres']   = movies['genres'].apply(parse_list)
movies['keywords'] = movies['keywords'].apply(parse_list)
movies['cast']     = movies['cast'].apply(parse_top_cast)
movies['director'] = movies['crew'].apply(parse_director)

print('Feature extraction done')
movies[['title', 'genres', 'cast', 'director']].head(3)


In [ ]:
# Remove spaces from multi-word names so they stay as single tokens
def collapse_spaces(lst):
    return [name.replace(' ', '') for name in lst]

movies['genres']   = movies['genres'].apply(collapse_spaces)
movies['keywords'] = movies['keywords'].apply(collapse_spaces)
movies['cast']     = movies['cast'].apply(collapse_spaces)
movies['director'] = movies['director'].apply(collapse_spaces)

# Tokenize overview
movies['overview'] = movies['overview'].apply(lambda x: x.split() if isinstance(x, str) else [])

# Director weighted x3 (most important signal)
# Cast weighted x2
movies['tags'] = (
    movies['overview'] +
    movies['genres'] +
    movies['keywords'] +
    movies['cast'] + movies['cast'] +
    movies['director'] + movies['director'] + movies['director']
)

print('Tags built. Sample:')
print(' '.join(movies['tags'].iloc[0][:30]), '...')


In [ ]:
# Genre frequency chart
all_genres = [g for sublist in movies['genres'] for g in sublist]
genre_counts = Counter(all_genres).most_common(15)

fig, ax = plt.subplots(figsize=(10, 5))
genres_names = [g[0] for g in genre_counts]
genres_vals  = [g[1] for g in genre_counts]
bars = ax.barh(genres_names, genres_vals,
               color=plt.cm.viridis(np.linspace(0.2, 0.9, len(genres_names))))
ax.set_title('Top 15 Genres in Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Movies')
ax.invert_yaxis()
for bar, val in zip(bars, genres_vals):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('models/genre_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Genre chart saved')


## Section 4 - TF-IDF Vectorization & Cosine Similarity

In [ ]:
# Build final dataframe
new_df = movies[['movie_id', 'title', 'tags', 'genres',
                  'vote_average', 'vote_count', 'popularity', 'release_date']].copy()

# Join tags into a single string
new_df['tags'] = new_df['tags'].apply(lambda x: ' '.join(x).lower())

# Extract year from release_date
new_df['year'] = pd.to_datetime(new_df['release_date'], errors='coerce').dt.year.fillna(0).astype(int)

# Genres as clean string for display
new_df['genres_str'] = movies['genres'].apply(lambda x: ', '.join(x))

# Short overview for display (first 200 chars)
new_df['overview_short'] = movies['overview'].apply(
    lambda x: ' '.join(x[:40]) + '...' if len(x) > 40 else ' '.join(x)
)

print(f'Final dataframe shape: {new_df.shape}')
new_df[['title', 'year', 'genres_str', 'vote_average']].head(5)


In [ ]:
# Stemming to normalize words
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def stem_text(text):
    return ' '.join([
        ps.stem(word) for word in text.split()
        if word not in stop_words
    ])

new_df['tags'] = new_df['tags'].apply(stem_text)
print('Stemming done')
print('Sample tags:', new_df['tags'].iloc[0][:200])


In [ ]:
# TF-IDF vectorization
# TF-IDF is better than CountVectorizer:
#   - Downweights very common words across all movies
#   - Upweights rare distinctive words
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),      # unigrams + bigrams
    min_df=2,                # ignore terms in fewer than 2 movies
    sublinear_tf=True        # apply log normalization to tf
)

vectors = tfidf.fit_transform(new_df['tags'])
print(f'TF-IDF matrix shape : {vectors.shape}')
print(f'Vocabulary size     : {len(tfidf.vocabulary_)}')


In [ ]:
# Cosine similarity matrix
print('Computing cosine similarity...')
similarity = cosine_similarity(vectors)
print(f'Similarity matrix shape: {similarity.shape}')
print(f'Sample similarity scores for movie 0: {sorted(similarity[0], reverse=True)[:6]}')


## Section 5 - Recommendation Function

In [ ]:
def recommend(movie_title, n=5, min_votes=50, rating_boost=True):
    """
    Recommend top N similar movies.

    Parameters:
        movie_title  : exact title string
        n            : number of recommendations
        min_votes    : filter out movies with fewer votes
        rating_boost : slightly boost higher-rated movies

    Returns:
        list of dicts with movie info
    """
    matches = new_df[new_df['title'].str.lower() == movie_title.lower()]
    if matches.empty:
        # Fuzzy fallback — find closest title
        from difflib import get_close_matches
        close = get_close_matches(movie_title, new_df['title'].tolist(), n=1, cutoff=0.5)
        if close:
            print(f'[INFO] Using closest match: {close[0]}')
            matches = new_df[new_df['title'] == close[0]]
        else:
            return []

    movie_index = matches.index[0]
    distances   = similarity[movie_index].copy()

    # Optional: multiply similarity by a small rating boost
    if rating_boost:
        norm_rating = new_df['vote_average'] / 10.0
        distances   = distances * (0.85 + 0.15 * norm_rating)

    # Sort by score, exclude the query movie itself
    ranked = sorted(
        [(i, score) for i, score in enumerate(distances) if i != movie_index],
        key=lambda x: x[1],
        reverse=True
    )

    results = []
    for idx, score in ranked:
        row = new_df.iloc[idx]
        if row['vote_count'] < min_votes:
            continue
        results.append({
            'title'         : row['title'],
            'year'          : int(row['year']),
            'genres'        : row['genres_str'],
            'rating'        : round(float(row['vote_average']), 1),
            'overview'      : row['overview_short'],
            'movie_id'      : int(row['movie_id']),
            'similarity_pct': round(score * 100, 1)
        })
        if len(results) == n:
            break
    return results


In [ ]:
# Test the recommender
print('=== Recommendations for Avatar ===')
for i, m in enumerate(recommend('Avatar'), 1):
    print(f'{i}. {m["title"]} ({m["year"]}) | Rating: {m["rating"]} | Match: {m["similarity_pct"]}%')
    print(f'   Genres: {m["genres"]}')
    print()

print('=== Recommendations for The Dark Knight ===')
for i, m in enumerate(recommend('The Dark Knight'), 1):
    print(f'{i}. {m["title"]} ({m["year"]}) | Rating: {m["rating"]} | Match: {m["similarity_pct"]}%')
    print()


## Section 6 - Save Artifacts

In [ ]:
import os
os.makedirs('models', exist_ok=True)

# Save enriched movie dataframe
pickle.dump(new_df.to_dict(), open('models/movie_dict.pkl', 'wb'))
print('Saved models/movie_dict.pkl')

# Save similarity matrix
pickle.dump(similarity, open('models/similarity.pkl', 'wb'))
print('Saved models/similarity.pkl')

# Save movie list for app dropdown
movie_list = new_df['title'].tolist()
pickle.dump(movie_list, open('models/movie_list.pkl', 'wb'))
print('Saved models/movie_list.pkl')

print(f'\nTotal movies in system: {len(movie_list)}')


In [ ]:
# Top rated movies in the dataset
print('Top 10 Highest Rated Movies (min 500 votes):')
top_rated = new_df[new_df['vote_count'] >= 500].nlargest(10, 'vote_average')[
    ['title', 'year', 'vote_average', 'genres_str']
]
print(top_rated.to_string(index=False))
